In [1]:
from database import postgres_connection

conn = postgres_connection()

In [2]:
import pandas as pd
import numpy as np

In [3]:
# data =  pd.read_sql(sql = 
# """select * from public.user_data u
# left join public.feed_data f on u.user_id=f.user_id
# left join public.post_text_df p on p.post_id = f.post_id
# where action = 'view' 
# order by u.user_id limit 2000000""", con = conn)

In [3]:
user_data = pd.read_sql(sql = 'select * from public.user_data LIMIT 5000000', con = conn)
feed_data = pd.read_sql(sql = "select * from public.feed_data where action = 'view' order by user_id LIMIT 5000000", con = conn)
post_text_df = pd.read_sql(sql = 'select * from public.post_text_df LIMIT 5000000', con = conn)

C:\Users\Max\AppData\Local\Temp\ipykernel_1144\2314649983.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  user_data = pd.read_sql(sql = 'select * from public.user_data LIMIT 5000000', con = conn)
C:\Users\Max\AppData\Local\Temp\ipykernel_1144\2314649983.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  feed_data = pd.read_sql(sql = "select * from public.feed_data where action = 'view' order by user_id LIMIT 5000000", con = conn)
C:\Users\Max\AppData\Local\Temp\ipykernel_1144\2314649983.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider 

In [5]:
user_topic_stats = pd.read_sql(sql = 
"""select user_id,topic, count(*) as topic_total_post_viewed, sum(target) as topic_total_post_liked from public.feed_data f
left join public.post_text_df p on p.post_id=f.post_id
where action='view'
group by user_id,topic""", con = conn)

C:\Users\Max\AppData\Local\Temp\ipykernel_1144\2304366955.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  user_topic_stats = pd.read_sql(sql =


In [6]:
total_post_stats = pd.read_sql(sql = 
"""select post_id, count(*) as total_post_viewed, sum(target) as total_post_liked from public.feed_data
where action='view'
group by post_id""", con = conn)

C:\Users\Max\AppData\Local\Temp\ipykernel_1144\1434119832.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  total_post_stats = pd.read_sql(sql =


In [7]:
total_post_stats.head()

,post_id,total_post_viewed,total_post_liked
0,1,8508,1067
1,2,7495,637
2,3,8412,1122
3,4,8158,1171
4,5,8583,1153


In [8]:
total_user_stats = pd.read_sql(sql ="""
select user_id, count(*) as total_user_viewed, sum(target) as total_user_liked from public.feed_data
where action='view'
group by user_id
""", con = conn)

C:\Users\Max\AppData\Local\Temp\ipykernel_1144\1339931296.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  total_user_stats = pd.read_sql(sql ="""


In [9]:
user_data.head()

,user_id,gender,age,country,city,exp_group,os,source
0,200,1,34,Russia,Degtyarsk,3,Android,ads
1,201,0,37,Russia,Abakan,0,Android,ads
2,202,1,17,Russia,Smolensk,4,Android,ads
3,203,0,18,Russia,Moscow,1,iOS,ads
4,204,0,36,Russia,Anzhero-Sudzhensk,3,Android,ads


In [10]:
user_data = pd.merge(user_data, total_user_stats, how = 'left', on = 'user_id')

In [11]:
user_data

,user_id,gender,age,country,city,exp_group,os,source,total_user_viewed,total_user_liked
0,200,1,34,Russia,Degtyarsk,3,Android,ads,358,43
1,201,0,37,Russia,Abakan,0,Android,ads,690,58
2,202,1,17,Russia,Smolensk,4,Android,ads,637,87
3,203,0,18,Russia,Moscow,1,iOS,ads,321,61
4,204,0,36,Russia,Anzhero-Sudzhensk,3,Android,ads,138,23
...,...,...,...,...,...,...,...,...,...,...
163200,168548,0,36,Russia,Kaliningrad,4,Android,organic,362,20
163201,168549,0,18,Russia,Tula,2,Android,organic,252,22
163202,168550,1,41,Russia,Yekaterinburg,4,Android,organic,369,38
163203,168551,0,38,Russia,Moscow,3,iOS,organic,477,48


In [12]:
user_topic_stats.head()

,user_id,topic,topic_total_post_viewed,topic_total_post_liked
0,200,business,28,2
1,200,covid,93,9
2,200,entertainment,14,4
3,200,movie,88,15
4,200,politics,58,4


In [13]:
pivoted_df = user_topic_stats.pivot(
    index='user_id', 
    columns='topic', 
    values='topic_total_post_liked'
)

# 2. Заполняем пропуски нулями (если пользователь ни разу не лайкнул какую-то тему)
pivoted_df = pivoted_df.fillna(0).astype(int)

# 3. Переименовываем колонки в формат "{topic}_total_post_liked"
pivoted_df.columns = [f"{col}_total_post_liked" for col in pivoted_df.columns]

# 4. Возвращаем user_id из индекса обратно в обычную колонку
result_df = pivoted_df.reset_index()

# Проверяем результат
result_df.head()

,user_id,business_total_post_liked,covid_total_post_liked,entertainment_total_post_liked,movie_total_post_liked,politics_total_post_liked,sport_total_post_liked,tech_total_post_liked
0,200,2,9,4,15,4,7,2
1,201,3,11,4,33,1,2,4
2,202,1,22,1,35,15,13,0
3,203,3,13,2,25,5,11,2
4,204,1,8,1,9,1,2,1


In [14]:
user_data = pd.merge(user_data, result_df, how = 'left', on = 'user_id')

In [15]:
user_data.head()

,user_id,gender,age,country,city,exp_group,os,source,total_user_viewed,total_user_liked,business_total_post_liked,covid_total_post_liked,entertainment_total_post_liked,movie_total_post_liked,politics_total_post_liked,sport_total_post_liked,tech_total_post_liked
0,200,1,34,Russia,Degtyarsk,3,Android,ads,358,43,2,9,4,15,4,7,2
1,201,0,37,Russia,Abakan,0,Android,ads,690,58,3,11,4,33,1,2,4
2,202,1,17,Russia,Smolensk,4,Android,ads,637,87,1,22,1,35,15,13,0
3,203,0,18,Russia,Moscow,1,iOS,ads,321,61,3,13,2,25,5,11,2
4,204,0,36,Russia,Anzhero-Sudzhensk,3,Android,ads,138,23,1,8,1,9,1,2,1


In [16]:
bins = [0, 16, 21, 25, 35, 45, 100]
labels = ['Under 17', '17-21 (Peak)', '22-25', '26-35', '36-45', '46+']

# Добавляем новую категориальную колонку
user_data['age_group'] = pd.cut(user_data['age'], bins=bins, labels=labels)
user_data['age_group'] = user_data['age_group'].astype(str)

# Проверяем, сколько активностей попало в каждую группу
print(user_data['age_group'].value_counts())

# # Находим список стран, чья доля в датасете выше 1%
# top_countries = user_data['country'].value_counts(normalize=True)
# top_countries = top_countries[top_countries >= 0.01].index.tolist()

# # Если страна не входит в топ, заменяем её на 'Other'
# user_data['country_clean'] = user_data['country'].apply(lambda x: x if x in top_countries else 'Other')

# # Проверяем что получилось
# print(user_data['country_clean'].value_counts(normalize=True) * 100)

user_data.drop('country', axis=1, inplace=True)

user_data.drop('city', axis=1, inplace=True)

user_data.drop('os', axis=1, inplace=True)

# user_data.drop('os', axis=1, inplace=True)

# user_data['source'] = user_data['source'].map({'ads': 0, 'organic': 1})
user_data.drop('source', axis=1, inplace=True)

age_group
17-21 (Peak)    47040
26-35           39169
22-25           29817
36-45           22135
Under 17        14190
46+             10854
Name: count, dtype: int64


In [17]:
user_data

,user_id,gender,age,exp_group,total_user_viewed,total_user_liked,business_total_post_liked,covid_total_post_liked,entertainment_total_post_liked,movie_total_post_liked,politics_total_post_liked,sport_total_post_liked,tech_total_post_liked,age_group
0,200,1,34,3,358,43,2,9,4,15,4,7,2,26-35
1,201,0,37,0,690,58,3,11,4,33,1,2,4,36-45
2,202,1,17,4,637,87,1,22,1,35,15,13,0,17-21 (Peak)
3,203,0,18,1,321,61,3,13,2,25,5,11,2,17-21 (Peak)
4,204,0,36,3,138,23,1,8,1,9,1,2,1,36-45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163200,168548,0,36,4,362,20,1,6,2,7,1,0,3,36-45
163201,168549,0,18,2,252,22,0,5,0,6,7,4,0,17-21 (Peak)
163202,168550,1,41,4,369,38,1,10,2,22,0,1,2,36-45
163203,168551,0,38,3,477,48,1,16,3,22,1,4,1,36-45


In [19]:
post_text_df.to_csv('posts.csv', index=False)

In [22]:
post_text_df

,post_id,text,topic
0,1,UK economy facing major risks\n\nThe UK manufa...,business
1,2,Aids and climate top Davos agenda\n\nClimate c...,business
2,3,Asian quake hits European shares\n\nShares in ...,business
3,4,India power shares jump on debut\n\nShares in ...,business
4,5,Lacroix label bought by US firm\n\nLuxury good...,business
...,...,...,...
7018,7315,"OK, I would not normally watch a Farrelly brot...",movie
7019,7316,I give this movie 2 stars purely because of it...,movie
7020,7317,I cant believe this film was allowed to be mad...,movie
7021,7318,The version I saw of this film was the Blockbu...,movie


In [23]:
post_text_df=pd.read_csv('post_text_df_with_embeddings.csv')

In [24]:
post_text_df

,post_id,text,topic,text_emb_0,text_emb_1,text_emb_2,text_emb_3,text_emb_4,text_emb_5,text_emb_6,text_emb_7,text_emb_8,text_emb_9,text_emb_10,text_emb_11,text_emb_12,text_emb_13,text_emb_14
0,1,UK economy facing major risks\n\nThe UK manufa...,business,1.940993,6.981473,1.697517,-3.901813,3.315840,1.809476,0.056912,2.848134,2.149449,-1.297853,2.073914,1.856352,1.589536,1.264085,-0.145595
1,2,Aids and climate top Davos agenda\n\nClimate c...,business,0.468049,6.347454,0.290504,-2.130036,0.125794,0.720989,0.353439,0.101012,0.842700,-1.577304,-0.925317,-0.019663,1.460197,0.096078,1.107126
2,3,Asian quake hits European shares\n\nShares in ...,business,2.279715,6.444628,2.251620,-2.190900,0.994169,4.152244,1.681589,0.583438,2.247523,-0.557069,-0.929320,1.465642,0.611004,0.148946,0.266726
3,4,India power shares jump on debut\n\nShares in ...,business,-0.548070,6.478297,3.026909,-3.530331,0.212879,2.093908,2.698010,4.277413,-0.312406,-0.700378,-1.775626,-1.607750,-1.928764,-2.103827,-1.166683
4,5,Lacroix label bought by US firm\n\nLuxury good...,business,-0.523950,6.052123,-2.654009,-0.883948,-0.720478,0.189525,1.602935,0.925413,-1.158416,0.375028,-1.648471,0.589777,-0.523645,0.073274,1.905876
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7018,7315,"OK, I would not normally watch a Farrelly brot...",movie,5.801983,-4.048192,-0.733327,1.479186,0.232272,-0.371871,-3.213414,1.797409,-1.680396,0.116428,-2.407177,-0.366477,0.806714,1.955927,0.509216
7019,7316,I give this movie 2 stars purely because of it...,movie,5.074275,-5.250196,1.331520,-0.782454,2.488329,1.050602,-0.696904,0.893325,-2.032128,1.627371,-0.880940,-0.025084,-1.285888,-2.684616,1.096504
7020,7317,I cant believe this film was allowed to be mad...,movie,2.773987,-4.532969,-0.461375,-1.222297,0.307661,0.740097,-3.225868,-0.276669,-1.500816,0.327532,-2.334728,1.279249,-0.451891,-0.475521,-0.350148
7021,7318,The version I saw of this film was the Blockbu...,movie,4.580049,-3.742197,-2.791309,-0.261278,0.975279,0.683044,-0.330054,1.192083,0.429272,0.076180,0.371762,-0.256304,-0.911360,0.469192,0.807353


In [ ]:
# adding bert embeddings - it was calculated on Kaggle
# from transformers import AutoTokenizer, AutoModel
# from sklearn.decomposition import PCA
# from sqlalchemy import create_engine
# from tqdm import tqdm
# import torch

# print("Загрузка предобученной модели трансформера из Hugging Face...")
# device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# # LaBSE отлично справляется как с английским, так и с русским текстом в одном датасете
# model_name = "sentence-transformers/LaBSE" 


# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModel.from_pretrained(model_name).to(device)
# model.eval()

# # 3. Функция для генерации эмбеддингов батчами (чтобы не перегрузить память)
# @torch.inference_mode()
# def get_embeddings(texts, batch_size=32):
#     embeddings = []
#     for i in tqdm(range(0, len(texts), batch_size), desc="Генерация эмбеддингов постов"):
#         batch_texts = texts[i:i+batch_size].tolist()
        
#         # Токенизация с обрезкой длинных текстов под лимит трансформера (512 токенов)
#         inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
#         inputs = {k: v.to(device) for k, v in inputs.items()}
        
#         outputs = model(**inputs)
#         # Берем [CLS] токен (индекс 0) как репрезентацию всего смысла текста
#         batch_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
#         embeddings.append(batch_emb)
        
#     return np.vstack(embeddings)

# # Запускаем извлечение векторов
# raw_embeddings = get_embeddings(post_text_df['text'].values)

# # 4. Сжатие признаков с помощью PCA (Снижение размерности с 768 до 15)
# print("Сжатие эмбеддингов с помощью PCA...")
# n_components = 15
# pca = PCA(n_components=n_components, random_state=1)
# compressed_embeddings = pca.fit_transform(raw_embeddings)

# # Создаем имена для новых колонок
# emb_cols = [f"text_emb_{i}" for i in range(n_components)]
# df_emb = pd.DataFrame(compressed_embeddings, columns=emb_cols)

# # Добавляем новые колонки в твой исходный DataFrame
# post_text_df = pd.concat([post_text_df, df_emb], axis=1)

# print("Расчет окончен! Новые колонки успешно добавлены в post_text_df.")
# print(post_text_df[emb_cols].head())

e:\python_projects\ML_course_project2\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Загрузка предобученной модели трансформера из Hugging Face...


e:\python_projects\ML_course_project2\env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Max\.cache\huggingface\hub\models--sentence-transformers--LaBSE. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
e:\python_projects\ML_course_project2\env\Lib\site-packages\huggingface_hub\file_download.py:149: Use

KeyboardInterrupt: 

In [25]:
post_text_df = pd.merge(post_text_df, total_post_stats, how = 'left', on = 'post_id')


In [26]:
post_text_df.head()

,post_id,text,topic,text_emb_0,text_emb_1,text_emb_2,text_emb_3,text_emb_4,text_emb_5,text_emb_6,text_emb_7,text_emb_8,text_emb_9,text_emb_10,text_emb_11,text_emb_12,text_emb_13,text_emb_14,total_post_viewed,total_post_liked
0,1,UK economy facing major risks\n\nThe UK manufa...,business,1.940993,6.981473,1.697517,-3.901813,3.315840,1.809476,0.056912,2.848134,2.149449,-1.297853,2.073914,1.856352,1.589536,1.264085,-0.145595,8508.0,1067.0
1,2,Aids and climate top Davos agenda\n\nClimate c...,business,0.468049,6.347454,0.290504,-2.130036,0.125794,0.720989,0.353439,0.101012,0.842700,-1.577304,-0.925317,-0.019663,1.460197,0.096078,1.107126,7495.0,637.0
2,3,Asian quake hits European shares\n\nShares in ...,business,2.279715,6.444628,2.251620,-2.190900,0.994169,4.152244,1.681589,0.583438,2.247523,-0.557069,-0.929320,1.465642,0.611004,0.148946,0.266726,8412.0,1122.0
3,4,India power shares jump on debut\n\nShares in ...,business,-0.548070,6.478297,3.026909,-3.530331,0.212879,2.093908,2.698010,4.277413,-0.312406,-0.700378,-1.775626,-1.607750,-1.928764,-2.103827,-1.166683,8158.0,1171.0
4,5,Lacroix label bought by US firm\n\nLuxury good...,business,-0.523950,6.052123,-2.654009,-0.883948,-0.720478,0.189525,1.602935,0.925413,-1.158416,0.375028,-1.648471,0.589777,-0.523645,0.073274,1.905876,8583.0,1153.0


In [27]:
# ==========================================
unique_texts = post_text_df['text'].dropna().drop_duplicates()

# Создаем временный DataFrame для уникальных постов, где и проведем все расчеты
text_metrics = unique_texts.to_frame()

# ==========================================
# ШАГ 2: Считаем метрики на маленьком датасете
# ==========================================

# 1. Длина текста (быстрый встроенный метод)
text_metrics['text_length'] = text_metrics['text'].str.len()

# 2. Количество слов (быстрый регулярный метод)
# Переводим в тип 'Int64' (с большой буквы), чтобы избежать проблем с типами данных
text_metrics['word_count'] = text_metrics['text'].str.split().str.len().astype('Int64')

# 3. Количество уникальных слов (считаем один раз для каждого уникального поста)
text_metrics['unique_word_count'] = text_metrics['text'].apply(lambda x: len(set(x.split()))).astype('Int64')

# ==========================================
# ШАГ 3: Быстро переносим (маппим) результаты на весь большой DataFrame data
# ==========================================

# Создаем словари «текст -> метрика» для мгновенного переноса данных
length_map = text_metrics.set_index('text')['text_length']
word_count_map = text_metrics.set_index('text')['word_count']
unique_word_map = text_metrics.set_index('text')['unique_word_count']

# Применяем словари ко всему огромному датасету (это займет доли секунды)
post_text_df['text_length'] = post_text_df['text'].map(length_map)
post_text_df['word_count'] = post_text_df['text'].map(word_count_map)
post_text_df['unique_word_count'] = post_text_df['text'].map(unique_word_map)

# Проверяем результат
print("Расчет окончен успешно!")
print(post_text_df[['text', 'text_length', 'word_count', 'unique_word_count']].head())


# from sklearn.feature_extraction.text import TfidfVectorizer
# # 1. Обучаем векторизатор только на УНИКАЛЬНЫХ текстах (экономим память)
# unique_posts = post_text_df[['text']].dropna().drop_duplicates().copy()

# # На практике лучше ограничить max_features (например, до 5000), 
# # чтобы не учитывать редкие опечатки и мусорные слова
# tfidf = TfidfVectorizer() 
# tfidf_matrix_unique = tfidf.fit_transform(unique_posts['text'])

# # 2. Считаем среднее значение TF-IDF для каждого уникального текста
# # axis=1 означает, что мы усредняем значения внутри каждой строки (для каждого поста)
# # .A1 превращает матрицу размера (N, 1) в обычный плоский одномерный массив NumPy
# unique_posts['mean_tfidf'] = tfidf_matrix_unique.mean(axis=1).A1

# # 3. Мгновенно переносим (маппим) готовые средние значения на ваш огромный датасет data
# text_to_tfidf_map = unique_posts.set_index('text')['mean_tfidf']
# post_text_df['mean_tfidf'] = post_text_df['text'].map(text_to_tfidf_map)

# print("Расчет среднего TF-IDF успешно завершен!")
# print(post_text_df[['text', 'mean_tfidf']].head())

Расчет окончен успешно!
                                                text  text_length  word_count  \
0  UK economy facing major risks\n\nThe UK manufa...         1967         324   
1  Aids and climate top Davos agenda\n\nClimate c...         2701         448   
2  Asian quake hits European shares\n\nShares in ...         3408         546   
3  India power shares jump on debut\n\nShares in ...         1026         173   
4  Lacroix label bought by US firm\n\nLuxury good...          889         150   

   unique_word_count  
0                197  
1                280  
2                329  
3                113  
4                104  


In [28]:
post_text_df

,post_id,text,topic,text_emb_0,text_emb_1,text_emb_2,text_emb_3,text_emb_4,text_emb_5,text_emb_6,...,text_emb_10,text_emb_11,text_emb_12,text_emb_13,text_emb_14,total_post_viewed,total_post_liked,text_length,word_count,unique_word_count
0,1,UK economy facing major risks\n\nThe UK manufa...,business,1.940993,6.981473,1.697517,-3.901813,3.315840,1.809476,0.056912,...,2.073914,1.856352,1.589536,1.264085,-0.145595,8508.0,1067.0,1967,324,197
1,2,Aids and climate top Davos agenda\n\nClimate c...,business,0.468049,6.347454,0.290504,-2.130036,0.125794,0.720989,0.353439,...,-0.925317,-0.019663,1.460197,0.096078,1.107126,7495.0,637.0,2701,448,280
2,3,Asian quake hits European shares\n\nShares in ...,business,2.279715,6.444628,2.251620,-2.190900,0.994169,4.152244,1.681589,...,-0.929320,1.465642,0.611004,0.148946,0.266726,8412.0,1122.0,3408,546,329
3,4,India power shares jump on debut\n\nShares in ...,business,-0.548070,6.478297,3.026909,-3.530331,0.212879,2.093908,2.698010,...,-1.775626,-1.607750,-1.928764,-2.103827,-1.166683,8158.0,1171.0,1026,173,113
4,5,Lacroix label bought by US firm\n\nLuxury good...,business,-0.523950,6.052123,-2.654009,-0.883948,-0.720478,0.189525,1.602935,...,-1.648471,0.589777,-0.523645,0.073274,1.905876,8583.0,1153.0,889,150,104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7018,7315,"OK, I would not normally watch a Farrelly brot...",movie,5.801983,-4.048192,-0.733327,1.479186,0.232272,-0.371871,-3.213414,...,-2.407177,-0.366477,0.806714,1.955927,0.509216,16940.0,2619.0,803,159,108
7019,7316,I give this movie 2 stars purely because of it...,movie,5.074275,-5.250196,1.331520,-0.782454,2.488329,1.050602,-0.696904,...,-0.880940,-0.025084,-1.285888,-2.684616,1.096504,6572.0,677.0,800,150,111
7020,7317,I cant believe this film was allowed to be mad...,movie,2.773987,-4.532969,-0.461375,-1.222297,0.307661,0.740097,-3.225868,...,-2.334728,1.279249,-0.451891,-0.475521,-0.350148,6803.0,731.0,636,111,89
7021,7318,The version I saw of this film was the Blockbu...,movie,4.580049,-3.742197,-2.791309,-0.261278,0.975279,0.683044,-0.330054,...,0.371762,-0.256304,-0.911360,0.469192,0.807353,6785.0,680.0,728,140,97


In [29]:
feed_data

,timestamp,user_id,post_id,action,target
0,2021-11-23 09:21:13,200,6842,view,0
1,2021-11-23 09:21:50,200,1125,view,0
2,2021-11-23 09:22:12,200,972,view,0
3,2021-11-23 09:23:03,200,1508,view,1
4,2021-11-23 09:24:52,200,1141,view,0
...,...,...,...,...,...
4999995,2021-12-17 11:26:35,12104,2893,view,0
4999996,2021-12-17 11:29:23,12104,3708,view,0
4999997,2021-12-17 11:29:33,12104,6336,view,0
4999998,2021-12-17 11:31:05,12104,1057,view,0


In [30]:
data = pd.merge(feed_data,user_data, how = 'left', on = 'user_id')
data = pd.merge(data, post_text_df, how = 'left', on = 'post_id')

In [31]:
data.head()

,timestamp,user_id,post_id,action,target,gender,age,exp_group,total_user_viewed,total_user_liked,...,text_emb_10,text_emb_11,text_emb_12,text_emb_13,text_emb_14,total_post_viewed,total_post_liked,text_length,word_count,unique_word_count
0,2021-11-23 09:21:13,200,6842,view,0,1,34,3,358,43,...,1.949536,-0.339953,-0.820944,1.542174,0.637518,6770.0,722.0,1428,228,156
1,2021-11-23 09:21:50,200,1125,view,0,1,34,3,358,43,...,3.698631,-1.262085,0.842075,1.014264,-0.216798,22205.0,2683.0,2443,391,224
2,2021-11-23 09:22:12,200,972,view,0,1,34,3,358,43,...,0.891255,1.065312,-1.402878,-0.135920,0.976727,22359.0,2745.0,4156,729,361
3,2021-11-23 09:23:03,200,1508,view,1,1,34,3,358,43,...,0.559162,-1.731010,2.049498,2.246067,-0.289067,22250.0,2920.0,2339,419,238
4,2021-11-23 09:24:52,200,1141,view,0,1,34,3,358,43,...,1.040365,-0.648809,0.814017,-1.066007,-0.362108,22291.0,2968.0,2952,514,282


In [32]:
data.groupby(['age','topic']).count()['user_id']

age  topic        
14   business          3485
     covid            19597
     entertainment     3020
     movie            32274
     politics          6973
                      ...  
78   entertainment       11
     movie               49
     politics            11
     sport               10
     tech                 9
Name: user_id, Length: 448, dtype: int64

In [33]:
pd.to_datetime(data['timestamp']).dt.hour.value_counts().sort_index()

timestamp
0          7
6     185690
7     296574
8     292825
9     288012
10    293479
11    293374
12    301302
13    292802
14    291495
15    289811
16    293428
17    290137
18    296422
19    297159
20    297764
21    298702
22    293585
23    107432
Name: count, dtype: int64

In [34]:
data['timestamp']=pd.to_datetime(data['timestamp'])

In [35]:
import pandas as pd

# 1. Группируем и находим топ-5 дней с наименьшей активностью
data['date']=data['timestamp'].dt.date
daily_activity = data.groupby('date').size()
lowest_days = daily_activity.sort_values().head(5)

# 2. Превращаем результат в DataFrame для удобства работы
df_lowest = lowest_days.reset_index(name='activity_count')

# 3. Переводим обратно в формат datetime, чтобы извлечь день недели
df_lowest['date'] = pd.to_datetime(df_lowest['date'])

# 4. Получаем номер дня недели (0 = Понедельник, 6 = Воскресенье)
df_lowest['weekday_num'] = df_lowest['date'].dt.weekday

# Словарь для перевода на русский язык
ru_weekdays = {
    0: 'Понедельник', 1: 'Вторник', 2: 'Среда', 
    3: 'Четверг', 4: 'Пятница', 5: 'Суббота', 6: 'Воскресенье'
}

# 5. Маппим названия дней недели
df_lowest['day_of_week'] = df_lowest['weekday_num'].map(ru_weekdays)

# Оставляем только нужные колонки для вывода
result = df_lowest[['date', 'day_of_week', 'activity_count']]
print(result.to_string(index=False))

      date day_of_week  activity_count
2021-12-14     Вторник           51322
2021-10-28     Четверг           51517
2021-11-17       Среда           51853
2021-10-11 Понедельник           52055
2021-10-23     Суббота           52417


In [36]:
# 1. Извлекаем номер дня недели (0 = Понедельник, 6 = Воскресенье)
data['weekday_num'] = data['timestamp'].dt.weekday

# 2. Группируем по номеру дня и считаем количество активностей
weekday_activity = data.groupby('weekday_num').size()

# Словарь для красивого отображения названий на русском
ru_weekdays = {
    0: 'Понедельник', 1: 'Вторник', 2: 'Среда', 
    3: 'Четверг', 4: 'Пятница', 5: 'Суббота', 6: 'Воскресенье'
}

# 3. Переименовываем индексы с чисел на названия дней недели
weekday_activity.index = weekday_activity.index.map(ru_weekdays)

# Выводим результат
print("Активность пользователей по дням недели:")
print(weekday_activity)

Активность пользователей по дням недели:
weekday_num
Понедельник    715209
Вторник        726699
Среда          717047
Четверг        668201
Пятница        719656
Суббота        720271
Воскресенье    732917
dtype: int64


In [37]:
data.drop('weekday_num', axis=1, inplace=True)

In [38]:
data['is_weekend']=data['timestamp'].dt.weekday.isin([5,6])
data['hour']=data['timestamp'].dt.hour
data['period_of_day'] = data['hour'].apply(lambda x: 'night' if x < 5 else ('morning' if x < 12 else ('afternoon' if x < 18 else 'evening')))

In [39]:
data.drop('timestamp', axis=1, inplace=True)

In [40]:
data.head()

,user_id,post_id,action,target,gender,age,exp_group,total_user_viewed,total_user_liked,business_total_post_liked,...,text_emb_14,total_post_viewed,total_post_liked,text_length,word_count,unique_word_count,date,is_weekend,hour,period_of_day
0,200,6842,view,0,1,34,3,358,43,2,...,0.637518,6770.0,722.0,1428,228,156,2021-11-23,False,9,morning
1,200,1125,view,0,1,34,3,358,43,2,...,-0.216798,22205.0,2683.0,2443,391,224,2021-11-23,False,9,morning
2,200,972,view,0,1,34,3,358,43,2,...,0.976727,22359.0,2745.0,4156,729,361,2021-11-23,False,9,morning
3,200,1508,view,1,1,34,3,358,43,2,...,-0.289067,22250.0,2920.0,2339,419,238,2021-11-23,False,9,morning
4,200,1141,view,0,1,34,3,358,43,2,...,-0.362108,22291.0,2968.0,2952,514,282,2021-11-23,False,9,morning


In [41]:
data.drop('date', axis=1, inplace=True)

In [42]:
data.head()

,user_id,post_id,action,target,gender,age,exp_group,total_user_viewed,total_user_liked,business_total_post_liked,...,text_emb_13,text_emb_14,total_post_viewed,total_post_liked,text_length,word_count,unique_word_count,is_weekend,hour,period_of_day
0,200,6842,view,0,1,34,3,358,43,2,...,1.542174,0.637518,6770.0,722.0,1428,228,156,False,9,morning
1,200,1125,view,0,1,34,3,358,43,2,...,1.014264,-0.216798,22205.0,2683.0,2443,391,224,False,9,morning
2,200,972,view,0,1,34,3,358,43,2,...,-0.135920,0.976727,22359.0,2745.0,4156,729,361,False,9,morning
3,200,1508,view,1,1,34,3,358,43,2,...,2.246067,-0.289067,22250.0,2920.0,2339,419,238,False,9,morning
4,200,1141,view,0,1,34,3,358,43,2,...,-1.066007,-0.362108,22291.0,2968.0,2952,514,282,False,9,morning


In [44]:
# data.drop('post_id', axis=1, inplace=True)

In [43]:
data.head()

,user_id,post_id,action,target,gender,age,exp_group,total_user_viewed,total_user_liked,business_total_post_liked,...,text_emb_13,text_emb_14,total_post_viewed,total_post_liked,text_length,word_count,unique_word_count,is_weekend,hour,period_of_day
0,200,6842,view,0,1,34,3,358,43,2,...,1.542174,0.637518,6770.0,722.0,1428,228,156,False,9,morning
1,200,1125,view,0,1,34,3,358,43,2,...,1.014264,-0.216798,22205.0,2683.0,2443,391,224,False,9,morning
2,200,972,view,0,1,34,3,358,43,2,...,-0.135920,0.976727,22359.0,2745.0,4156,729,361,False,9,morning
3,200,1508,view,1,1,34,3,358,43,2,...,2.246067,-0.289067,22250.0,2920.0,2339,419,238,False,9,morning
4,200,1141,view,0,1,34,3,358,43,2,...,-1.066007,-0.362108,22291.0,2968.0,2952,514,282,False,9,morning


In [46]:
# data[data['country']=='Russia'].groupby(['country','city']).count()['user_id']

In [48]:
# bins = [0, 16, 21, 25, 35, 45, 100]
# labels = ['Under 17', '17-21 (Peak)', '22-25', '26-35', '36-45', '46+']

# # Добавляем новую категориальную колонку
# data['age_group'] = pd.cut(data['age'], bins=bins, labels=labels)
# data['age_group'] = data['age_group'].astype(str)

# # Проверяем, сколько активностей попало в каждую группу
# print(data['age_group'].value_counts())

In [44]:
data.head()

,user_id,post_id,action,target,gender,age,exp_group,total_user_viewed,total_user_liked,business_total_post_liked,...,text_emb_13,text_emb_14,total_post_viewed,total_post_liked,text_length,word_count,unique_word_count,is_weekend,hour,period_of_day
0,200,6842,view,0,1,34,3,358,43,2,...,1.542174,0.637518,6770.0,722.0,1428,228,156,False,9,morning
1,200,1125,view,0,1,34,3,358,43,2,...,1.014264,-0.216798,22205.0,2683.0,2443,391,224,False,9,morning
2,200,972,view,0,1,34,3,358,43,2,...,-0.135920,0.976727,22359.0,2745.0,4156,729,361,False,9,morning
3,200,1508,view,1,1,34,3,358,43,2,...,2.246067,-0.289067,22250.0,2920.0,2339,419,238,False,9,morning
4,200,1141,view,0,1,34,3,358,43,2,...,-1.066007,-0.362108,22291.0,2968.0,2952,514,282,False,9,morning


In [45]:
data.columns

Index(['user_id', 'post_id', 'action', 'target', 'gender', 'age', 'exp_group',
       'total_user_viewed', 'total_user_liked', 'business_total_post_liked',
       'covid_total_post_liked', 'entertainment_total_post_liked',
       'movie_total_post_liked', 'politics_total_post_liked',
       'sport_total_post_liked', 'tech_total_post_liked', 'age_group', 'text',
       'topic', 'text_emb_0', 'text_emb_1', 'text_emb_2', 'text_emb_3',
       'text_emb_4', 'text_emb_5', 'text_emb_6', 'text_emb_7', 'text_emb_8',
       'text_emb_9', 'text_emb_10', 'text_emb_11', 'text_emb_12',
       'text_emb_13', 'text_emb_14', 'total_post_viewed', 'total_post_liked',
       'text_length', 'word_count', 'unique_word_count', 'is_weekend', 'hour',
       'period_of_day'],
      dtype='object')

In [46]:
# data.drop('text', axis=1, inplace=True)
data.drop('action', axis=1, inplace=True)

In [47]:
data.head()

,user_id,post_id,target,gender,age,exp_group,total_user_viewed,total_user_liked,business_total_post_liked,covid_total_post_liked,...,text_emb_13,text_emb_14,total_post_viewed,total_post_liked,text_length,word_count,unique_word_count,is_weekend,hour,period_of_day
0,200,6842,0,1,34,3,358,43,2,9,...,1.542174,0.637518,6770.0,722.0,1428,228,156,False,9,morning
1,200,1125,0,1,34,3,358,43,2,9,...,1.014264,-0.216798,22205.0,2683.0,2443,391,224,False,9,morning
2,200,972,0,1,34,3,358,43,2,9,...,-0.135920,0.976727,22359.0,2745.0,4156,729,361,False,9,morning
3,200,1508,1,1,34,3,358,43,2,9,...,2.246067,-0.289067,22250.0,2920.0,2339,419,238,False,9,morning
4,200,1141,0,1,34,3,358,43,2,9,...,-1.066007,-0.362108,22291.0,2968.0,2952,514,282,False,9,morning


In [48]:
data['is_weekend'] = data['is_weekend'].astype(int)

In [49]:
from sklearn.model_selection import train_test_split

data_ = data.drop(['target','user_id','post_id','text'], axis=1)
target = data[['target']]

X_train, X_test, y_train, y_test = train_test_split(data_,
                                                    target, 
                                                    test_size=0.25,
                                                    random_state=1)

In [50]:
print(data['topic'].value_counts(normalize=True) * 100)

topic
movie            36.93388
covid            24.17318
sport            13.33694
politics         10.13202
business          6.29070
entertainment     5.05818
tech              4.07510
Name: proportion, dtype: float64


In [51]:
X_train['age_group'].describe()

count          3750000
unique               6
top       17-21 (Peak)
freq           1108061
Name: age_group, dtype: object

In [52]:
category_columns = X_train.select_dtypes(include=['object']).columns.to_list()

In [53]:
category_columns

['age_group', 'topic', 'period_of_day']

In [54]:
X_train.select_dtypes(include=['object']).columns

Index(['age_group', 'topic', 'period_of_day'], dtype='object')

In [55]:
num_columns = X_train.columns.difference(category_columns).to_list()

In [56]:
num_columns

['age',
 'business_total_post_liked',
 'covid_total_post_liked',
 'entertainment_total_post_liked',
 'exp_group',
 'gender',
 'hour',
 'is_weekend',
 'movie_total_post_liked',
 'politics_total_post_liked',
 'sport_total_post_liked',
 'tech_total_post_liked',
 'text_emb_0',
 'text_emb_1',
 'text_emb_10',
 'text_emb_11',
 'text_emb_12',
 'text_emb_13',
 'text_emb_14',
 'text_emb_2',
 'text_emb_3',
 'text_emb_4',
 'text_emb_5',
 'text_emb_6',
 'text_emb_7',
 'text_emb_8',
 'text_emb_9',
 'text_length',
 'total_post_liked',
 'total_post_viewed',
 'total_user_liked',
 'total_user_viewed',
 'unique_word_count',
 'word_count']

In [57]:
data.isna().sum()

user_id                           0
post_id                           0
target                            0
gender                            0
age                               0
exp_group                         0
total_user_viewed                 0
total_user_liked                  0
business_total_post_liked         0
covid_total_post_liked            0
entertainment_total_post_liked    0
movie_total_post_liked            0
politics_total_post_liked         0
sport_total_post_liked            0
tech_total_post_liked             0
age_group                         0
text                              0
topic                             0
text_emb_0                        0
text_emb_1                        0
text_emb_2                        0
text_emb_3                        0
text_emb_4                        0
text_emb_5                        0
text_emb_6                        0
text_emb_7                        0
text_emb_8                        0
text_emb_9                  

In [58]:
for col in category_columns:
    X_train[col]=X_train[col].fillna(X_train[col].mode()[0]).astype(str)
    X_test[col]=X_test[col].fillna(X_test[col].mode()[0]).astype(str)
    # data[col]=data[col].fillna(data[col].mode()[0]).astype(str)

In [59]:
for col in num_columns:
    X_train[col]=X_train[col].fillna(0)
    X_test[col]=X_test[col].fillna(0)
    # data[col]=data[col].fillna(data[col].median())

In [60]:
num_cols_idx = [list(X_train.columns).index(col) for col in num_columns]
cat_column_ohe_idx = [list(X_train.columns).index(col) for col in category_columns]
# cat_column_ohe_idx = [list(X_train.columns).index(col) for col in category_columns if X_train[col].nunique() < 5]
# cat_column_target_idx = [list(X_train.columns).index(col) for col in category_columns if X_train[col].nunique() >= 5]

In [61]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from category_encoders import TargetEncoder
from category_encoders.one_hot import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, f1_score, roc_curve, auc

t = [('StandardScaler', StandardScaler(), num_cols_idx),
     ('OneHotEncoder', OneHotEncoder(), cat_column_ohe_idx),
    #  ('TargetEncoder', TargetEncoder(), cat_column_target_idx)
    ]

column_transfor = ColumnTransformer(transformers=t)


In [62]:
X_train.columns

Index(['gender', 'age', 'exp_group', 'total_user_viewed', 'total_user_liked',
       'business_total_post_liked', 'covid_total_post_liked',
       'entertainment_total_post_liked', 'movie_total_post_liked',
       'politics_total_post_liked', 'sport_total_post_liked',
       'tech_total_post_liked', 'age_group', 'topic', 'text_emb_0',
       'text_emb_1', 'text_emb_2', 'text_emb_3', 'text_emb_4', 'text_emb_5',
       'text_emb_6', 'text_emb_7', 'text_emb_8', 'text_emb_9', 'text_emb_10',
       'text_emb_11', 'text_emb_12', 'text_emb_13', 'text_emb_14',
       'total_post_viewed', 'total_post_liked', 'text_length', 'word_count',
       'unique_word_count', 'is_weekend', 'hour', 'period_of_day'],
      dtype='object')

In [63]:
pipe_lr = Pipeline([
    ('column_transfor', column_transfor),
    ('logistic_regression', LogisticRegression(
        penalty='l2',  C=5.0,  solver='lbfgs', n_jobs=-1,max_iter=300,))
])

pipe_lr.fit(X_train, y_train)
y_pred = pipe_lr.predict(X_test)
y_pred_proba = pipe_lr.predict_proba(X_test)[:, 1]

print(f"log regression score = {pipe_lr.score(X_test,y_test):.3f}")
print(classification_report(y_test, y_pred))

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

print(f"Значение AUC: {roc_auc:.4f}")


e:\python_projects\ML_course_project2\env\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


log regression score = 0.880
              precision    recall  f1-score   support

           0       0.88      1.00      0.94   1100736
           1       0.32      0.00      0.00    149264

    accuracy                           0.88   1250000
   macro avg       0.60      0.50      0.47   1250000
weighted avg       0.81      0.88      0.82   1250000

Значение AUC: 0.6529


In [64]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc

pipe_gb = Pipeline([
    ('column_transfor', column_transfor),
    ('gradient_boosting', GradientBoostingClassifier(
        learning_rate=1,    # Снижаем до 0.1 (шаг 1.0 заставляет модель метаться и строить лишнее)
        n_estimators=100,      # Сокращаем количество деревьев со 100 до 50 (для теста скорости)
        max_depth=5,          # Жестко ограничиваем глубину дерева (по умолчанию 3, не увеличивайте её!)
        
        # --- ГЛАВНЫЕ ПАРАМЕТРЫ ДЛЯ УСКОРЕНИЯ ---
        subsample=0.1,        # Обучаем каждое дерево только на 10% случайных строк (стохастический бустинг).
                              # Для 3.75 млн строк это гигантское ускорение без потери качества!
        # max_features='sqrt',  # Каждое дерево ищет сплиты не по всем 115+ колонкам, а по случайному корню из них
        
        random_state=1
    ))
])

# Обучаем модель (с ravel() для убирания предупреждений)
pipe_gb.fit(X_train, y_train.values.ravel())

# y_pred = pipe_gb.predict(X_test)
# y_pred_proba = pipe_gb.predict_proba(X_test)[:, 1]

# print(f"gradient boosting score = {pipe_gb.score(X_test,y_test):.3f}")
# print(classification_report(y_test, y_pred))

# fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
# roc_auc = auc(fpr, tpr)

# print(f"Значение AUC: {roc_auc:.4f}")

,steps,"[('column_transfor', ...), ('gradient_boosting', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('StandardScaler', ...), ('OneHotEncoder', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [65]:
y_pred = pipe_gb.predict(X_test)
y_pred_proba = pipe_gb.predict_proba(X_test)[:, 1]

# Считаем метрики (быстрый расчет на основе готовых y_pred)
print(f"gradient boosting score = {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
print(f"Значение AUC: {roc_auc:.4f}")

gradient boosting score = 0.877
              precision    recall  f1-score   support

           0       0.88      0.99      0.93   1100736
           1       0.25      0.01      0.03    149264

    accuracy                           0.88   1250000
   macro avg       0.56      0.50      0.48   1250000
weighted avg       0.81      0.88      0.83   1250000

Значение AUC: 0.7015


In [66]:
from sklearn.ensemble import HistGradientBoostingClassifier

pipe_hgb = Pipeline([
    ('column_transfor', column_transfor),
    ('gradient_boosting', HistGradientBoostingClassifier(
        # 1. Постепенное обучение
        learning_rate=0.05,       # Снижение шага до 0.05 или 0.03 заставляет модель точнее настраивать веса
        max_iter=250,             # Увеличиваем число итераций, так как шаг стал меньше
        
        # 2. Усложнение структуры деревьев
        max_depth=7,              # Позволяет улавливать более сложные нелинейные зависимости и взаимодействия
        max_leaf_nodes=63,        # Должно коррелировать с max_depth (обычно в пределах от 2^(depth-1) до 2^depth)
        min_samples_leaf=50,      # Защита от переобучения на мелких подвыборках (по умолчанию 20, можно поднять до 50-100)
        
        # 3. Добавление регуляризации
        l2_regularization=0.1,    # L2-регуляризация штрафует слишком большие веса в листьях, спасает от оверфиттинга
        
        # 4. Контроль остановки
        early_stopping=True,
        n_iter_no_change=15,      # Даем модели чуть больше «времени» (15 итераций вместо 10) на выход из локального плато
        validation_fraction=0.1,  # 10% данных для валидации early stopping
        
        scoring='roc_auc',        # Обучение и остановка будут ориентироваться строго на вашу целевую метрику!
        random_state=1
    ))
])
pipe_hgb.fit(X_train, y_train.values.ravel())

,steps,"[('column_transfor', ...), ('gradient_boosting', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('StandardScaler', ...), ('OneHotEncoder', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [67]:
y_pred = pipe_hgb.predict(X_test)
y_pred_proba = pipe_hgb.predict_proba(X_test)[:, 1]

# Считаем метрики (быстрый расчет на основе готовых y_pred)
print(f"gradient boosting score = {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
print(f"Значение AUC: {roc_auc:.4f}")

gradient boosting score = 0.881
              precision    recall  f1-score   support

           0       0.88      1.00      0.94   1100736
           1       0.00      0.00      0.00    149264

    accuracy                           0.88   1250000
   macro avg       0.44      0.50      0.47   1250000
weighted avg       0.78      0.88      0.82   1250000

Значение AUC: 0.7305


In [68]:
from sklearn.ensemble import HistGradientBoostingClassifier
categorical_cols = ['topic', 'age_group', 'period_of_day']
for col in categorical_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype('category')
    if col in X_test.columns:
        X_test[col] = X_test[col].astype('category')

pipe_hgb_m = Pipeline([
    ('gradient_boosting', HistGradientBoostingClassifier(
        categorical_features='from_dtype',  # Модель сама найдет колонки с типом 'category' (запятая исправлена)
        
        # 1. Постепенное обучение
        learning_rate=0.05,       
        max_iter=250,             
        
        # 2. Усложнение структуры деревьев
        max_depth=7,              
        max_leaf_nodes=63,        
        min_samples_leaf=50,      
        
        # 3. Добавление регуляризации
        l2_regularization=0.1,    
        
        # 4. Контроль остановки
        early_stopping=True,
        n_iter_no_change=15,      
        validation_fraction=0.1,  
        
        scoring='roc_auc',        
        random_state=1
    ))
])

# 4. Обучаем модель на очищенных данных
pipe_hgb_m.fit(X_train, y_train.values.ravel())

,steps,"[('gradient_boosting', ...)]"
,transform_input,None
,memory,None
,verbose,False
,loss,'log_loss'
,learning_rate,0.05
,max_iter,250
,max_leaf_nodes,63
,max_depth,7
,min_samples_leaf,50
,l2_regularization,0.1


In [69]:
y_pred = pipe_hgb_m.predict(X_test)
y_pred_proba = pipe_hgb_m.predict_proba(X_test)[:, 1]

# Считаем метрики (быстрый расчет на основе готовых y_pred)
print(f"gradient boosting score = {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
print(f"Значение AUC: {roc_auc:.4f}")

gradient boosting score = 0.881
              precision    recall  f1-score   support

           0       0.88      1.00      0.94   1100736
           1       0.50      0.00      0.00    149264

    accuracy                           0.88   1250000
   macro avg       0.69      0.50      0.47   1250000
weighted avg       0.84      0.88      0.82   1250000

Значение AUC: 0.7308


In [99]:
import pickle

with open("pipe_hgb_dl.pkl", "wb") as f:
    pickle.dump(pipe_hgb_m, f)

# # Загрузка модели из файла
# with open("model.pkl", "rb") as f:
#     loaded_model = pickle.load(f)

# # Проверим, что работает
# print(loaded_model.predict_proba(X))

In [71]:
data.head()

,user_id,post_id,target,gender,age,exp_group,total_user_viewed,total_user_liked,business_total_post_liked,covid_total_post_liked,...,text_emb_13,text_emb_14,total_post_viewed,total_post_liked,text_length,word_count,unique_word_count,is_weekend,hour,period_of_day
0,200,6842,0,1,34,3,358,43,2,9,...,1.542174,0.637518,6770.0,722.0,1428,228,156,0,9,morning
1,200,1125,0,1,34,3,358,43,2,9,...,1.014264,-0.216798,22205.0,2683.0,2443,391,224,0,9,morning
2,200,972,0,1,34,3,358,43,2,9,...,-0.135920,0.976727,22359.0,2745.0,4156,729,361,0,9,morning
3,200,1508,1,1,34,3,358,43,2,9,...,2.246067,-0.289067,22250.0,2920.0,2339,419,238,0,9,morning
4,200,1141,0,1,34,3,358,43,2,9,...,-1.066007,-0.362108,22291.0,2968.0,2952,514,282,0,9,morning


In [72]:
user_data.head()

,user_id,gender,age,exp_group,total_user_viewed,total_user_liked,business_total_post_liked,covid_total_post_liked,entertainment_total_post_liked,movie_total_post_liked,politics_total_post_liked,sport_total_post_liked,tech_total_post_liked,age_group
0,200,1,34,3,358,43,2,9,4,15,4,7,2,26-35
1,201,0,37,0,690,58,3,11,4,33,1,2,4,36-45
2,202,1,17,4,637,87,1,22,1,35,15,13,0,17-21 (Peak)
3,203,0,18,1,321,61,3,13,2,25,5,11,2,17-21 (Peak)
4,204,0,36,3,138,23,1,8,1,9,1,2,1,36-45


In [73]:


df_user_features = user_data.copy()
# df_user_features = data[user_cols].copy()

print(f"Таблица пользователей готова. Уникальных профилей: {df_user_features.shape[0]}")


Таблица пользователей готова. Уникальных профилей: 163205


In [74]:
print(df_user_features.isna().sum())

user_id                           0
gender                            0
age                               0
exp_group                         0
total_user_viewed                 0
total_user_liked                  0
business_total_post_liked         0
covid_total_post_liked            0
entertainment_total_post_liked    0
movie_total_post_liked            0
politics_total_post_liked         0
sport_total_post_liked            0
tech_total_post_liked             0
age_group                         0
dtype: int64


In [75]:
post_text_df

,post_id,text,topic,text_emb_0,text_emb_1,text_emb_2,text_emb_3,text_emb_4,text_emb_5,text_emb_6,...,text_emb_10,text_emb_11,text_emb_12,text_emb_13,text_emb_14,total_post_viewed,total_post_liked,text_length,word_count,unique_word_count
0,1,UK economy facing major risks\n\nThe UK manufa...,business,1.940993,6.981473,1.697517,-3.901813,3.315840,1.809476,0.056912,...,2.073914,1.856352,1.589536,1.264085,-0.145595,8508.0,1067.0,1967,324,197
1,2,Aids and climate top Davos agenda\n\nClimate c...,business,0.468049,6.347454,0.290504,-2.130036,0.125794,0.720989,0.353439,...,-0.925317,-0.019663,1.460197,0.096078,1.107126,7495.0,637.0,2701,448,280
2,3,Asian quake hits European shares\n\nShares in ...,business,2.279715,6.444628,2.251620,-2.190900,0.994169,4.152244,1.681589,...,-0.929320,1.465642,0.611004,0.148946,0.266726,8412.0,1122.0,3408,546,329
3,4,India power shares jump on debut\n\nShares in ...,business,-0.548070,6.478297,3.026909,-3.530331,0.212879,2.093908,2.698010,...,-1.775626,-1.607750,-1.928764,-2.103827,-1.166683,8158.0,1171.0,1026,173,113
4,5,Lacroix label bought by US firm\n\nLuxury good...,business,-0.523950,6.052123,-2.654009,-0.883948,-0.720478,0.189525,1.602935,...,-1.648471,0.589777,-0.523645,0.073274,1.905876,8583.0,1153.0,889,150,104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7018,7315,"OK, I would not normally watch a Farrelly brot...",movie,5.801983,-4.048192,-0.733327,1.479186,0.232272,-0.371871,-3.213414,...,-2.407177,-0.366477,0.806714,1.955927,0.509216,16940.0,2619.0,803,159,108
7019,7316,I give this movie 2 stars purely because of it...,movie,5.074275,-5.250196,1.331520,-0.782454,2.488329,1.050602,-0.696904,...,-0.880940,-0.025084,-1.285888,-2.684616,1.096504,6572.0,677.0,800,150,111
7020,7317,I cant believe this film was allowed to be mad...,movie,2.773987,-4.532969,-0.461375,-1.222297,0.307661,0.740097,-3.225868,...,-2.334728,1.279249,-0.451891,-0.475521,-0.350148,6803.0,731.0,636,111,89
7021,7318,The version I saw of this film was the Blockbu...,movie,4.580049,-3.742197,-2.791309,-0.261278,0.975279,0.683044,-0.330054,...,0.371762,-0.256304,-0.911360,0.469192,0.807353,6785.0,680.0,728,140,97


In [76]:

df_post_features = post_text_df.copy()
# df_post_features = data[post_cols].copy()


print(f"Таблица постов готова. Уникальных постов: {df_post_features.shape[0]}")

Таблица постов готова. Уникальных постов: 7023


In [77]:
print(df_post_features.isna().sum())

post_id                0
text                   0
topic                  0
text_emb_0             0
text_emb_1             0
text_emb_2             0
text_emb_3             0
text_emb_4             0
text_emb_5             0
text_emb_6             0
text_emb_7             0
text_emb_8             0
text_emb_9             0
text_emb_10            0
text_emb_11            0
text_emb_12            0
text_emb_13            0
text_emb_14            0
total_post_viewed    192
total_post_liked     192
text_length            0
word_count             0
unique_word_count      0
dtype: int64


In [78]:
df_post_features.fillna(0, inplace=True)

In [ ]:
YOUR_LOGIN = 'maksim-zatsepin-eyb5957' 
import os
from dotenv import load_dotenv
load_dotenv('.env')

conn_str = PG_CONSTR= os.environ['PG_CONSTR']

# 1. Сохраняем признаки пользователей
df_user_features.to_sql(
    f"{YOUR_LOGIN}_user_features_dl",
    conn_str,
    if_exists="replace",
    index=False,
    method="multi"
)
print("Таблица пользователей успешно улетела в БД!")

# 2. Сохраняем признаки постов
df_post_features.to_sql(
    f"{YOUR_LOGIN}_post_features_dl",
    conn_str,
    if_exists="replace",
    index=False,
    method="multi"
)
print("Таблица постов успешно улетела в БД!")

Таблица пользователей успешно улетела в БД!
Таблица постов успешно улетела в БД!


In [81]:
X_train.columns

Index(['gender', 'age', 'exp_group', 'total_user_viewed', 'total_user_liked',
       'business_total_post_liked', 'covid_total_post_liked',
       'entertainment_total_post_liked', 'movie_total_post_liked',
       'politics_total_post_liked', 'sport_total_post_liked',
       'tech_total_post_liked', 'age_group', 'topic', 'text_emb_0',
       'text_emb_1', 'text_emb_2', 'text_emb_3', 'text_emb_4', 'text_emb_5',
       'text_emb_6', 'text_emb_7', 'text_emb_8', 'text_emb_9', 'text_emb_10',
       'text_emb_11', 'text_emb_12', 'text_emb_13', 'text_emb_14',
       'total_post_viewed', 'total_post_liked', 'text_length', 'word_count',
       'unique_word_count', 'is_weekend', 'hour', 'period_of_day'],
      dtype='object')

# NEXT PART is expirimental part - it was not used finally due to hitrate was too low and model was not accepted by the system. It was used for testing and experiments only.

In [91]:
# adding new model
# Финальный максимизированный скрипт обучения нейросети
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

# 1. Подготовка списков колонок
CAT_COLS = ['topic', 'period_of_day']
NUM_COLS = [
    col
    for col in X_train.columns
    if col not in CAT_COLS and col != 'age_group'
]

In [92]:
# 2. Расчет циклических признаков времени
X_train['hour_sin'] = np.sin(2 * np.pi * X_train['hour'] / 24.0)
X_train['hour_cos'] = np.cos(2 * np.pi * X_train['hour'] / 24.0)

X_test['hour_sin'] = np.sin(2 * np.pi * X_test['hour'] / 24.0)
X_test['hour_cos'] = np.cos(2 * np.pi * X_test['hour'] / 24.0)

# Обновляем список числовых колонок
if 'hour' in NUM_COLS:
    NUM_COLS.remove('hour')
NUM_COLS.extend(['hour_sin', 'hour_cos'])

In [94]:
# 3. Объявление классов архитектуры PyTorch
class RecDataset(torch.utils.data.Dataset):

    def __init__(self, X_num, X_cat, y=None):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        if y is not None:
            self.y = torch.tensor(y.values, dtype=torch.float32)
        else:
            self.y = None

    def __len__(self):
        return len(self.X_num)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X_num[idx], self.X_cat[idx], self.y[idx]
        return self.X_num[idx], self.X_cat[idx]


class DeepRecommenderNet(nn.Module):

    def __init__(self, num_features_dim, cat_dims, emb_dims):
        super().__init__()
        self.embeddings = nn.ModuleList(
            [
                nn.Embedding(num_classes, emb_dim)
                for num_classes, emb_dim in zip(cat_dims, emb_dims)
            ]
        )

        total_input_dim = num_features_dim + sum(emb_dims)

        self.mlp = nn.Sequential(
            nn.Linear(total_input_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),  # Современная и гладкая функция активации
            nn.Dropout(0.15),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x_num, x_cat):
        emb_outputs = [
            emb_layer(x_cat[:, i])
            for i, emb_layer in enumerate(self.embeddings)
        ]
        x = torch.cat([x_num] + emb_outputs, dim=1)
        return self.mlp(x)


class SklearnNNWrapper:

    def __init__(self, model_path, num_features_dim, cat_dims, emb_dims):
        self.model = DeepRecommenderNet(num_features_dim, cat_dims, emb_dims)
        self.model.load_state_dict(torch.load(model_path, map_location='cpu'))
        self.model.eval()

    def predict_proba(self, X_num, X_cat):
        X_num_tensor = torch.tensor(X_num, dtype=torch.float32)
        X_cat_tensor = torch.tensor(X_cat, dtype=torch.long)
        with torch.no_grad():
            prob_class_1 = self.model(X_num_tensor, X_cat_tensor).numpy().flatten()
        return np.column_stack((1.0 - prob_class_1, prob_class_1))


In [ ]:

# 4. Кодирование категориальных признаков в индексы
cat_mappings = {}
for col in CAT_COLS:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

    mapping = {val: idx for idx, val in enumerate(X_train[col].unique())}
    cat_mappings[col] = mapping

    X_train[col] = X_train[col].map(mapping).fillna(0).astype(int)
    X_test[col] = X_test[col].map(mapping).fillna(0).astype(int)

with open("cat_mappings.pkl", "wb") as f:
    pickle.dump(cat_mappings, f)

# 5. Масштабирование признаков СТРОГО после фиксации финального NUM_COLS
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[NUM_COLS])
X_test_num = scaler.transform(X_test[NUM_COLS])

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

X_train_cat = X_train[CAT_COLS].values
X_test_cat = X_test[CAT_COLS].values

# 6. Подготовка DataLoader
train_dataset = RecDataset(X_train_num, X_train_cat, y_train)
train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True)

# 7. Инициализация и запуск обучения улучшенной нейросети
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
cat_dims = [X_train['topic'].nunique(), X_train['period_of_day'].nunique()]
emb_dims = [32, 16]  # Увеличенная емкость скрытых векторов для максимизации качества
num_features_dim = len(NUM_COLS)

model = DeepRecommenderNet(num_features_dim, cat_dims, emb_dims).to(device)
criterion = nn.BCELoss()
# Переключаемся на AdamW с легкой регуляризацией весов (weight_decay)
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)

model.train()
print("Старт обучения нейросети...")
for epoch in range(3):
    epoch_loss = 0
    for batch_num, batch_cat, batch_y in train_loader:
        batch_num, batch_cat, batch_y = (
            batch_num.to(device),
            batch_cat.to(device),
            batch_y.to(device),
        )

        optimizer.zero_grad()

        # ПРАВИЛЬНО: убран .squeeze(), размерности идеально сходятся
        predictions = model(batch_num, batch_cat)
        loss = criterion(predictions, batch_y.float())

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(
        f"Эпоха {epoch+1} завершена. Средний Loss: {epoch_loss/len(train_loader):.4f}"
    )

# 8. Упаковка весов и сборка финального единого pkl-файла артефактов
torch.save(model.cpu().state_dict(), "nn_weights.pt")

wrapper = SklearnNNWrapper("nn_weights.pt", num_features_dim, cat_dims, emb_dims)

# artifacts = {
#     'model': wrapper,
#     'scaler': scaler,
#     'cat_mappings': cat_mappings,
#     'num_columns': NUM_COLS,
# }

# with open("pipe_hgb_dl.pkl", "wb") as f:
#     pickle.dump(artifacts, f)
# print(
#     "Все артефакты упакованы и финальный пайплайн успешно сохранен в 'pipe_hgb_dl.pkl'!"
# )

Старт обучения нейросети...
Эпоха 1 завершена. Средний Loss: 0.3626
Эпоха 2 завершена. Средний Loss: 0.3335
Эпоха 3 завершена. Средний Loss: 0.3317
Все артефакты упакованы и финальный пайплайн успешно сохранен в 'pipe_hgb_dl.pkl'!


In [96]:
from sklearn.metrics import roc_auc_score

# 1. Извлекаем модель-обертку из нашего словаря артефактов


# 2. Передаем в метод predict_proba отмасштабированные числа и индексы категорий
# Метод возвращает матрицу из двух колонок: [вероятность_0, вероятность_1]
# Нам нужна вторая колонка [:, 1] — это и есть вероятность лайка (y_pred_proba)
y_pred_proba = wrapper.predict_proba(X_test_num, X_test_cat)[:, 1]

# 3. Считаем итоговый ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC нейросети на тестовой выборке: {roc_auc:.4f}")

ROC-AUC нейросети на тестовой выборке: 0.7283


In [98]:
import pickle
import torch

print("Сборка единого словаря артефактов...")

# 1. Извлекаем чистые веса (state_dict) нейросети в память
network_weights = model.state_dict()

# 2. Собираем ВСЕ компоненты (включая веса PyTorch) в один стандартный словарь Python
artifacts = {
    'scaler': scaler,
    'cat_mappings': cat_mappings,
    'num_columns': NUM_COLS,
    'model_weights': network_weights  # Веса нейросети теперь внутри словаря!
}

# 3. Сохраняем ОДИН файл model.pkl, который примет система
with open("model-dl.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("Успешно! Все артефакты и веса нейросети упакованы в ОДИН файл 'model-dl.pkl'.")


Сборка единого словаря артефактов...
Успешно! Все артефакты и веса нейросети упакованы в ОДИН файл 'model-dl.pkl'.
